# Manual lineup scenario

Use this notebook to test optimizer behavior without hitting Yahoo. Adjust players, starting flags, and positions, then re-run the plan cells.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lineup import apply_plan_to_roster, optimize_lineup, render_plan, render_roster
from models import Player, RosterSnapshot


In [2]:
roster = RosterSnapshot(
    team_key='422.l.100000.t.3',
    team_name='Daily Test Team',
    lineup_date='2026-03-17',
    coverage_type='date',
    players=[
        Player(
            player_key='422.p.1001',
            player_id='1001',
            name='Bench Bat',
            editorial_team_abbr='LAD',
            editorial_team_full_name='Los Angeles Dodgers',
            display_position='OF',
            primary_position='OF',
            eligible_positions=('OF', 'Util'),
            selected_position='BN',
            is_starting_today=True,
        ),
        Player(
            player_key='422.p.1002',
            player_id='1002',
            name='Cold Starter',
            editorial_team_abbr='BOS',
            editorial_team_full_name='Boston Red Sox',
            display_position='OF',
            primary_position='OF',
            eligible_positions=('OF', 'Util'),
            selected_position='OF',
            is_starting_today=False,
        ),
        Player(
            player_key='422.p.1003',
            player_id='1003',
            name='Locked Starter',
            editorial_team_abbr='ATL',
            editorial_team_full_name='Atlanta Braves',
            display_position='1B',
            primary_position='1B',
            eligible_positions=('1B', 'Util'),
            selected_position='1B',
            is_starting_today=False,
            is_locked=True,
        ),
        Player(
            player_key='422.p.1004',
            player_id='1004',
            name='Everyday Catcher',
            editorial_team_abbr='SEA',
            editorial_team_full_name='Seattle Mariners',
            display_position='C',
            primary_position='C',
            eligible_positions=('C',),
            selected_position='C',
            is_starting_today=True,
        ),
    ],
)

projections = {
    '422.p.1001': 8.2,
    '422.p.1002': 4.5,
    '422.p.1004': 7.9,
}

print(render_roster(roster))


Team: Daily Test Team
Date: 2026-03-17
Roster:
- 1B   Locked Starter (1B) [not starting]
- C    Everyday Catcher (C) [starting]
- OF   Cold Starter (OF) [not starting]
- BN   Bench Bat (OF) [starting]


In [3]:
plan = optimize_lineup(roster, projections=projections)
print(render_plan(plan))


Plan:
- Cold Starter: OF -> BN (Not starting today; swapped out for Bench Bat.)
- Bench Bat: BN -> OF (Starting today and eligible at OF.)


In [4]:
updated_roster = apply_plan_to_roster(roster, plan)
print(render_roster(updated_roster))


Team: Daily Test Team
Date: 2026-03-17
Roster:
- 1B   Locked Starter (1B) [not starting]
- C    Everyday Catcher (C) [starting]
- OF   Bench Bat (OF) [starting]
- BN   Cold Starter (OF) [not starting]
